# Notes

The `pem` files are in `ASN.1` format with `DER` encoding which is basiacally `tag,length,value`

This is the offical [RSA](https://www.rfc-editor.org/rfc/rfc8017) Private Key sequence 

```
      RSAPrivateKey ::= SEQUENCE {
          version           Version,
          modulus           INTEGER,  -- n
          publicExponent    INTEGER,  -- e
          privateExponent   INTEGER,  -- d
          prime1            INTEGER,  -- p
          prime2            INTEGER,  -- q
          exponent1         INTEGER,  -- d mod (p-1)
          exponent2         INTEGER,  -- d mod (q-1)
          coefficient       INTEGER,  -- (inverse of q) mod p
          otherPrimeInfos   OtherPrimeInfos OPTIONAL
      }
```

```
   o  version is the version number, for compatibility with future
      revisions of this document.  It SHALL be 0 for this version of the
      document, unless multi-prime is used; in which case, it SHALL be
      1.

            Version ::= INTEGER { two-prime(0), multi(1) }
               (CONSTRAINED BY
               {-- version must be multi if otherPrimeInfos present --})
   o  modulus is the RSA modulus n.
   o  publicExponent is the RSA public exponent e.
   o  privateExponent is the RSA private exponent d.
   o  prime1 is the prime factor p of n.
   o  prime2 is the prime factor q of n.
   o  exponent1 is d mod (p - 1).
   o  exponent2 is d mod (q - 1).
   o  coefficient is the CRT coefficient q^(-1) mod p.
```

However, there are two differences between RSA [Formats](https://stackoverflow.com/questions/20065304/differences-between-begin-rsa-private-key-and-begin-private-key):

- `BEGIN PRIVATE KEY` -> PKCS#8 just base64 encoded 
- `BEGIN RSA PRIVATE KEY` -> PKCS#1 which maps to the above DER ASN.1 sequence

In [7]:
from base64 import b64decode

from asn1 import Decoder,Encoder
import asn1
from Crypto.Util.number import long_to_bytes,bytes_to_long

In [3]:
data1 = b64decode(
r"""
MIIJKAIBAAKCAgEAoovj1+p3c8duZu7Bq91nblqfR973/Sbtkcsq22GxTJsFYfS8
LpVTgYX67Mu94Jvezivo6UsNNV3J+8tqqDHjylgfhLq1KNgCGjASTrPv8w17ogvU
0mcYbBdnBQUgX6HgmfnfDoNROcN+M5xTbGxWsmJPbLzqyS991Mt30ysNyqX3Xu+m
IcMndkDUeUpwOK86Ane6WLgmYS2rxjyoiyKB6uASZkKVoUlxNO/L089j/eatStc+
2FLOI1/+IbJf32lYO+hP/13vJTtsynjeFAobIHasJuU+mxdAqN1iGYCXSv9CzL8Z
zkjybDujXY0P4kAh285q04jqNxpmS1XWTOxnPgz747eyE7w7SMcFsQQzMRu7+Fmm
x8rknAp8gG1W4A6Grh8xoRq1N4kgeF/L8VT7LG+z1o1o+uMj6CDuS1F2LKWt9U8A
ZgWAALX2LU19irgE3xTy7tEHFumTMRMu3kw/dy7lHdOf5L1K0DZu7KIvEyOXcd5K
OTyo1whgizctSXNuIEjMZsDcDreYu3XW+KQq+7i5BbANyRaWETiJi0Vj/5MPYgB/
yHq9VPBwt66aXmYTRX3ZzxU/GGXbCsjlVVv2Z4bofFc+P/nktBKQ7ctLUWJaarM5
IHqFhiwEX+07LylushSguCcp98ch3XbZRbW5nzIuJeqE4K1pb1g4eyq1FSsCAwEA
AQKCAgA2
""" + "=="
)

data2 = b64decode(
r"""
QQKCAQEAvp0qAvuY
Fqbfz7SS9eOEvt0Av49SBiGyJ+07JPRI3RNpV2DfpQtjpWjz7yWQ35GR9eyWb4n+
McUezGVZGx3438lsEF5ox5fiYCD9lea2p5OL/sHLTD6k8WSpsHJVObP/lB+HiW+4
0jbg+f3V8CyOwm7tEhdGWuQCli0PGrbp8SMGQSC89/NynJRGw48fARJh71u5/pUK
RXONYqHRgApBapPpnhjtVg4/tV9LsvE/uK4C6QbJiiMp0v2pg7jsmoDtQdjSz9jx
""" + "=="
)


In [4]:
fields = [
"version",
"modulus",
"publicExponent",
"privateExponent",
"prime1",
"prime2",
"exponent1",
"exponent2",
"coefficient",
"otherPrimeInfos",
]

In [5]:
decoder = Decoder()
decoder.start(data1)

tag = decoder.peek()
print(tag)
decoder.enter()

rsa = dict()

for field in fields:
    try:
        tag,value = decoder.read()
        rsa[field] = value
        print(tag)
    except Exception as e:
        print(e)
        break

rsa

Tag(nr=<Numbers.Sequence: 0x10>, typ=<Types.Constructed: 0x20>, cls=<Classes.Universal: 0x00>)
Tag(nr=<Numbers.Integer: 0x02>, typ=<Types.Primitive: 0x00>, cls=<Classes.Universal: 0x00>)
Tag(nr=<Numbers.Integer: 0x02>, typ=<Types.Primitive: 0x00>, cls=<Classes.Universal: 0x00>)
Tag(nr=<Numbers.Integer: 0x02>, typ=<Types.Primitive: 0x00>, cls=<Classes.Universal: 0x00>)
ASN1 decoding error: premature end of input.


{'version': 0,
 'modulus': 6631316416098838682802128343990275479532872786651633539948307190697908603373175042943029087469126335369551901080272168603377528062774915375165530918519630395903999699636499285743849913885324803882792524331343085107400361455206535178173026652842551667010792740141148574669923241716685840700567395563695683814638479664554204129149343739157420569111004511751638261693380535053508524077389507177203389328868994767756036799113319442378424352750557390285360968500617892853097966841680931551468742493893473019692284099608290395487589688820135234040006300479343362031873077308950625037014102294847164403322851328633204831296994307911879106084671297572247955460728732539288478468077816461758938324545468930325525874778066617263923632419576172889885680100941891523592108795263444596833565731820263723335534834254009932851282094049243829332784858614272674306859617419541721171200464219125880571531582657687816225361326759388253525010283830526176716469851143674762476365336930188804862

In [11]:
for i in range(len(data2)):
    try:
        decoder = Decoder()
        decoder.start(data2[i:])

        tag,value = decoder.read()
        if tag.nr == asn1.Numbers.Integer:
            print(tag,value)
    except:
        pass

Tag(nr=2, typ=<Types.Primitive: 0x00>, cls=<Classes.Context: 0x80>) b'\x01'


In [43]:
# Let's just guess what the last value could be, I am guessing prime1/prime2 
assert value % rsa["modulus"] == 0, "Its not prime1 or prime2"

AssertionError: Its not prime1 or prime2

In [41]:
import Crypto.PublicKey.RSA as RSA
from Crypto.Cipher import PKCS1_v1_5

In [ ]:
# exponent1 is d mod (p - 1)
""" 
Known:
    n,e,d mod (p-1) [either p or q] 

    d' = d % (p-1)


"""


with open("encrypted.txt","rb") as f:
    print(
        PKCS1_v1_5.new(key).decrypt(f.read(),None)
    )

bytearray(b'CTF{learning_the_der_encoding_helps}')
